# Settimana 24 — Chiusura del piano: portfolio, primo cliente e roadmap

**Obiettivo:** trasformare 24 settimane di lavoro in una presenza pubblica pronta per i clienti.
Questo notebook produce i deliverable finali: audit del portfolio,
generatore di proposta di valore, lista di contatti prioritizzati, piano d'azione
con date reali e `progress.md` aggiornato con il riepilogo dell'intero piano.

**Costo stimato:** < $0.05 (un paio di chiamate a gpt-4o-mini per la proposta di valore
e il riepilogo esecutivo del portfolio).

**Qui non c'è codice di infrastruttura.** Tutto quello che hai costruito nelle settimane precedenti
c'è già. Questa settimana il lavoro è di strategia e comunicazione.

## Cella 0 — Setup

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai", "python-dotenv"])

import os
from pathlib import Path
from datetime import date, timedelta
from dotenv import load_dotenv

CURSO_DIR   = Path("../").resolve()
PROJECT_DIR = CURSO_DIR / "proyectos"
LAUNCH_DIR  = CURSO_DIR / "lanzamiento"
LAUNCH_DIR.mkdir(exist_ok=True)

env_path = PROJECT_DIR / "research-agent" / ".env"
if env_path.exists():
    load_dotenv(env_path)

from openai import OpenAI
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
assert os.getenv("OPENAI_API_KEY"), "Definisci OPENAI_API_KEY"

HOY = date.today()
print(f"✓ Directory di lancio: {LAUNCH_DIR}")
print(f"✓ Data: {HOY.isoformat()}")

# Inventario del corso completato
dispensas = sorted((CURSO_DIR / "dispensas").glob("*.md"))
notebooks = sorted((CURSO_DIR / "notebooks").glob("*.ipynb"))
print(f"\nInventario del corso:")
print(f"  Dispense:   {len(dispensas)} / 24")
print(f"  Notebook:   {len(notebooks)} / 24")

## Cella 1 — Audit del portfolio

Rispondi alle quattro domande del cliente per ogni progetto.

In [ ]:
PROYECTOS_PORTFOLIO = [
    {
        "nombre":   "DocBot RAG API",
        "repo":     "github.com/tu-usuario/docbot-api",
        "problema": "Le aziende del settore agroalimentare perdono ore a cercare "
                    "informazioni in centinaia di documenti interni (normativa, report, "
                    "certificazioni) che nessuno legge perché non esiste un modo efficiente di consultarli.",
        "para_quien": "Reparti qualità, R&S e normativa in aziende medie "
                      "del settore agroalimentare con più di 500 documenti tecnici interni.",
        "valor_dolor": "2-4 ore settimanali a persona in ricerche manuali. "
                       "Rischio di non conformità normativa per mancata conoscenza degli aggiornamenti.",
        "diferenciador": "Costruito specificamente per la documentazione tecnica del settore "
                         "ittico e agroalimentare galiziano. Funziona in gallego e spagnolo. "
                         "Distribuibile on-premise per dati sensibili.",
        # Metriche reali del progetto
        "metricas": {
            "latencia_p95_ms": 850,
            "precision_respuestas": 0.91,
            "documentos_indexados_demo": 47,
            "coste_por_consulta_eur": 0.002,
        },
    },
    {
        "nombre":   "ResearchAgent API",
        "repo":     "github.com/tu-usuario/research-agent",
        "problema": "I team di innovazione dedicano 30-60 minuti a ogni ricerca "
                    "di tendenze tecnologiche, raccogliendo fonti manualmente prima "
                    "di poter scrivere un report.",
        "para_quien": "Team di innovazione, scouting tecnologico e trasferimento "
                      "tecnologico in cluster settoriali, EDIH e centri tecnologici.",
        "valor_dolor": "3-5 ore settimanali a persona in ricerca e sintesi manuale. "
                       "Report che richiedono giorni per essere generati quando il mercato cambia in poche ore.",
        "diferenciador": "Agente autonomo con Plan-and-Execute + ReAct. Cerca su internet "
                         "e nel corpus documentale del cliente nello stesso processo. "
                         "API asincrona con osservabilità completa e guardrail di sicurezza.",
        "metricas": {
            "tiempo_informe_s": 94,
            "tiempo_manual_equivalente_min": 45,
            "tasa_exito": 0.96,
            "coste_por_informe_eur": 0.008,
        },
    },
]

print("AUDIT DEL PORTFOLIO")
print("=" * 60)
for p in PROYECTOS_PORTFOLIO:
    print(f"\n### {p['nombre']}")
    print(f"  Repo:          {p['repo']}")
    print(f"  Problema:      {p['problema'][:80]}...")
    print(f"  Per chi:       {p['para_quien'][:80]}...")
    print(f"  Valore/dolore: {p['valor_dolor'][:80]}...")
    print(f"  Differenziale: {p['diferenciador'][:80]}...")
    print(f"  Metriche:      {p['metricas']}")

## Cella 2 — Generatore di proposta di valore con LLM

A partire dai dati del portfolio, l'LLM genera variazioni della proposta di valore.
Tu scegli e adatti quella che suona meglio.

In [ ]:
PERFIL = {
    "nombre":   "Raffaele Bini",
    "nicho":    "settore agroalimentare, ittico e biotecnologico in Galizia e Spagna",
    "contexto": "Lavora presso DIH DATAlife, un Hub di Innovazione Digitale Europeo specializzato "
                "in biorisorse, sostenibilità e salute. Conosce il settore dall'interno.",
    "capacidades": [
        "Agenti LLM autonomi per ricerca e sintesi di informazioni",
        "Sistemi RAG per la consultazione di documentazione interna",
        "Classificatori ed estrattori di informazioni da documenti settoriali",
        "API di IA pronte per la produzione con sicurezza e osservabilità",
    ],
    "diferenciador_clave": (
        "Non vende tecnologia generica: capisce i problemi specifici del settore "
        "(normativa ittica, certificazioni agroalimentari, documentazione tecnica forestale) "
        "e costruisce soluzioni che si integrano in quei flussi di lavoro reali."
    ),
}

def generar_propuestas_valor(perfil: dict, n: int = 4) -> list[str]:
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{
            "role": "user",
            "content": f"""Genera {n} variazioni di proposta di valore per questo profilo di consulente.
Ogni variazione deve seguire la struttura:
"Aiuto [tipo di azienda] a [risultato concreto] usando [meccanismo], senza [ostacolo]."

Requisiti:
- Niente gergo tecnico (non menzionare LLM, RAG, vector DB, ecc.)
- Specifico per la nicchia, non generico
- Una sola frase per variazione
- Ogni variazione su una riga, numerata

PROFILO:
Nicchia: {perfil['nicho']}
Contesto: {perfil['contexto']}
Capacità: {', '.join(perfil['capacidades'])}
Differenziale: {perfil['diferenciador_clave']}"""
        }],
        temperature=0.7,
        max_tokens=400,
    )
    texto = resp.choices[0].message.content.strip()
    return [l.strip() for l in texto.split("\n") if l.strip()]


print("Generazione delle proposte di valore...")
propuestas = generar_propuestas_valor(PERFIL)

print("\nPROPOSTE DI VALORE GENERATE:")
print("=" * 60)
for p in propuestas:
    print(f"  {p}")

print()
print("→ Scegli quella che suona meglio, adattala con le tue parole e salvala.")
print("  È la frase che dirai su LinkedIn, su Malt e nelle prime email.")

## Cella 3 — Lista di contatti Tier 1 prioritizzata

I primi clienti arrivano quasi sempre dalla rete immediata.
Compila questa lista con nomi reali e ordinala per probabilità di conversione.

In [ ]:
# Compila questa lista con i tuoi contatti reali
# priorità: 1 = molto probabile, 2 = possibile, 3 = lungo termine
CONTACTOS_TIER1 = [
    # Enti partner di DIH DATAlife — hai già una relazione
    {"nombre": "[Contatto ANFACO-CYTMA]",   "empresa": "ANFACO-CYTMA",   "rol": "Innovazione",   "prioridad": 1, "angulo": "Classificazione e ricerca nella documentazione tecnica ittica"},
    {"nombre": "[Contatto BIOGA]",           "empresa": "BIOGA",          "rol": "Direzione",     "prioridad": 1, "angulo": "Scouting tecnologico automatizzato per il settore biotech"},
    {"nombre": "[Contatto GRADIANT]",        "empresa": "GRADIANT",       "rol": "R&S",           "prioridad": 1, "angulo": "Strumenti LLM per progetti di ricerca"},
    {"nombre": "[Contatto ITG]",             "empresa": "ITG",            "rol": "Trasferimento", "prioridad": 2, "angulo": "Report automatizzati di innovazione per il settore forestale"},
    {"nombre": "[Contatto CETGA]",           "empresa": "CETGA",          "rol": "Tecnico",       "prioridad": 2, "angulo": "Agente di ricerca per l'acquacoltura"},
    # Contatti personali fuori dall'ecosistema
    {"nombre": "[Ex collega 1]",             "empresa": "[Azienda]",      "rol": "[Ruolo]",       "prioridad": 1, "angulo": "[Quale problema di IA potrebbe avere]"},
    {"nombre": "[Ex collega 2]",             "empresa": "[Azienda]",      "rol": "[Ruolo]",       "prioridad": 2, "angulo": "[Quale problema di IA potrebbe avere]"},
    {"nombre": "[Contatto università]",      "empresa": "USC / UDC / Uvigo","rol": "Ricerca",     "prioridad": 3, "angulo": "Automazione della revisione della letteratura scientifica"},
    {"nombre": "[Contatto settore privato 1]","empresa": "[Azienda]",     "rol": "[Ruolo]",       "prioridad": 2, "angulo": "[Quale problema di IA potrebbe avere]"},
    {"nombre": "[Contatto settore privato 2]","empresa": "[Azienda]",     "rol": "[Ruolo]",       "prioridad": 3, "angulo": "[Quale problema di IA potrebbe avere]"},
]

# Ordina per priorità
contactos_ordenados = sorted(CONTACTOS_TIER1, key=lambda x: x["prioridad"])

print("LISTA DI CONTATTI TIER 1 — per priorità di contatto")
print(f"{'#':<3} {'P':<3} {'Nome':<30} {'Azienda':<20} {'Angolo di ingresso'}")
print("-" * 100)
for i, c in enumerate(contactos_ordenados, 1):
    estrellas = "●" * c["prioridad"] + "○" * (3 - c["prioridad"])
    print(f"{i:<3} {estrellas:<5} {c['nombre']:<28} {c['empresa']:<20} {c['angulo'][:45]}")

print()
tier1_count = sum(1 for c in CONTACTOS_TIER1 if c["prioridad"] == 1)
print(f"Contatti priorità 1 (azione immediata): {tier1_count}")
print()
print("→ Sostituisci i placeholder con nomi reali prima della settimana 1 di lancio.")

## Cella 4 — Piano d'azione con date reali

In [ ]:
def semana(offset_semanas: int) -> str:
    """Restituisce la data di inizio della settimana N da oggi."""
    return (HOY + timedelta(weeks=offset_semanas)).strftime("%d %b %Y")


PLAN_LANZAMIENTO = [
    {
        "fase": "Settimana 1-2 — Portfolio pronto",
        "inicio": semana(0),
        "tareas": [
            "Aggiornare il README di DocBot e ResearchAgent con le 6 domande del cliente",
            "Pubblicare entrambi i progetti su GitHub con demo funzionanti",
            "Creare una pagina di presentazione (Notion, GitHub Pages o simile)",
            "Completare il profilo Malt con casi d'uso concreti",
            "Scrivere la proposta di valore definitiva (scelta nella cella precedente)",
        ],
    },
    {
        "fase": "Settimana 2-3 — Primi contatti",
        "inicio": semana(1),
        "tareas": [
            "Inviare un messaggio ai 5 contatti di priorità 1 (template della settimana 19)",
            "Prima pubblicazione su LinkedIn (problema del settore, non tecnologia)",
            "Contattare almeno 2 enti partner di DIH DATAlife con una proposta concreta",
            "Attivare il profilo Malt e fare la prima ricerca di progetti attivi",
        ],
    },
    {
        "fase": "Settimana 3-6 — Prime conversazioni",
        "inicio": semana(2),
        "tareas": [
            "Riunione di discovery con almeno 2 contatti interessati",
            "Inviare la prima proposta usando il template della settimana 19",
            "Seconda e terza pubblicazione su LinkedIn",
            "Completare la lista Tier 1 con nomi reali",
            "Rivedere il day rate in base alle conversazioni reali",
        ],
    },
    {
        "fase": "Mese 2 — Primo progetto",
        "inicio": semana(4),
        "tareas": [
            "Primo progetto firmato (pilota, ambito ridotto o progetto interno)",
            "Contratto firmato e anticipo incassato prima di iniziare",
            "Tracking del tempo reale fin dal primo giorno (per migliorare le stime future)",
            "Pubblicazione settimanale su LinkedIn stabilita come routine",
        ],
    },
    {
        "fase": "Mese 3 — Consegna e caso di successo",
        "inicio": semana(8),
        "tareas": [
            "Consegna del primo progetto con i tre documenti (settimana 22)",
            "Richiedere una testimonianza scritta e il permesso per un caso studio",
            "Pubblicare il caso studio nel portfolio",
            "Proporre una fase 2 o un retainer se il cliente è soddisfatto",
            "Rivedere le stime: quanto ci è voluto realmente rispetto a quanto stimato?",
        ],
    },
    {
        "fase": "Mese 4-6 — Consolidamento",
        "inicio": semana(13),
        "tareas": [
            "Secondo e terzo progetto completati",
            "Day rate adeguato con dati reali",
            "Decisione su un'eventuale specializzazione aggiuntiva",
            "Primo cliente ricorrente (retainer o progetto 2)",
            "Revisione della nicchia: è ancora quella giusta?",
        ],
    },
]

print(f"PIANO DI LANCIO — da {HOY.strftime('%d %b %Y')}")
print("=" * 60)
for fase in PLAN_LANZAMIENTO:
    print(f"\n📅 {fase['fase']} (a partire dal {fase['inicio']})")
    for t in fase["tareas"]:
        print(f"   ☐ {t}")

## Cella 5 — Generatore del README di vendita finale

Genera il README di presentazione del portfolio completo
(non di un singolo progetto, ma di te come consulente).

In [ ]:
def generar_readme_consultor(perfil: dict, proyectos: list, propuesta_valor: str) -> str:
    """
    Genera il README principale del portfolio.
    È la prima cosa che legge un potenziale cliente aprendo il profilo GitHub.
    """
    proyectos_txt = ""
    for p in proyectos:
        m = p["metricas"]
        metricas_str = " · ".join(f"{k}: {v}" for k, v in list(m.items())[:2])
        proyectos_txt += f"""
### [{p['nombre']}]({p['repo']})
{p['problema'][:120]}...

**Per chi:** {p['para_quien'][:80]}...
**Metriche:** {metricas_str}
"""

    capacidades = "\n".join(f"- {c}" for c in perfil["capacidades"])

    return f"""# {perfil['nombre']} — Consulente IA per il settore agroalimentare

> {propuesta_valor}

---

## Perché io e non un altro

{perfil['diferenciador_clave']}

---

## Cosa costruisco

{capacidades}

---

## Progetti
{proyectos_txt}

---

## Come lavoro

- Progetto pilota con ambito definito → consegna in 2-4 settimane
- Prezzo fisso o tempo e materiale a seconda dell'incertezza del progetto
- Documentazione completa: tecnica, guida d'uso e riepilogo esecutivo
- Supporto post-consegna incluso per le prime due settimane

---

## Parliamone

Hai un problema di gestione delle informazioni, classificazione di documenti
o automazione della ricerca nella tua azienda?

📧 **{perfil['nombre'].lower().replace(' ', '.')}@dihdatalife.com**
💼 **[Malt](#)** · **[LinkedIn](#)**

*Rispondo entro 24 ore nei giorni lavorativi.*
"""


# Usa la prima proposta generata come placeholder (adattala con quella che hai scelto)
propuesta_elegida = propuestas[0] if propuestas else (
    "Aiuto le aziende del settore agroalimentare a ridurre il tempo di ricerca "
    "e classificazione delle informazioni usando agenti di IA, "
    "senza bisogno di un team tecnico interno."
)

readme_consultor = generar_readme_consultor(PERFIL, PROYECTOS_PORTFOLIO, propuesta_elegida)
(LAUNCH_DIR / "README_portfolio.md").write_text(readme_consultor)
print("✓ lanzamiento/README_portfolio.md generato")
print()
print(readme_consultor[:600], "...")

## Cella 6 — `progress.md` finale: riepilogo delle 24 settimane

In [ ]:
RESUMEN_SEMANAS = [
    ("01", "Python avanzato per l'IA",                "Tipi, dataclass, async/await, pathlib, ambiente"),
    ("02", "API e OpenAI SDK",                        "Chat completions, streaming, function calling, costi"),
    ("03", "Prompt engineering",                       "Few-shot, CoT, structured output, system prompt"),
    ("04", "Embedding e ricerca semantica",            "text-embedding-3, cosine similarity, FAISS"),
    ("05", "Database vettoriali",                      "Chroma, indicizzazione, retrieval con metadati"),
    ("06", "Progetto 1 — DocBot RAG (base)",           "Chunking, ingestione, pipeline RAG completa"),
    ("07", "DocBot RAG — miglioramenti",               "Reranking, HyDE, valutazione con RAGAS"),
    ("08", "DocBot RAG — API FastAPI",                 "Endpoint, Pydantic, streaming, test"),
    ("09", "Valutazione di sistemi RAG",               "RAGAS, LLM-as-judge, metriche faithfulness/relevance"),
    ("10", "Agenti LLM — fondamenti",                  "ReAct, tool calling, ciclo di ragionamento"),
    ("11", "Agenti LLM — strumenti",                   "Function calling, tool definition, gestione errori"),
    ("12", "Agenti multi-step",                        "Plan-and-Execute, memoria, stato della ricerca"),
    ("13", "Strategia freelance (base)",               "Nicchia, day rate, canali, portfolio come lettera di vendita"),
    ("14", "Deployment cloud",                         "Railway/Render/Fly.io, CI/CD, HEALTHCHECK, smoke test"),
    ("15", "Progetto 2 — ResearchAgent",               "Plan-Execute+ReAct completo, DuckDuckGo, Chroma, sintesi"),
    ("16", "API asincrona dell'agente",                "Fire-and-poll, JobStore, 202/409/204, test mock"),
    ("17", "LLMOps e osservabilità",                   "Tracer, tiktoken, log JSON, costo per trace, dashboard"),
    ("18", "Langfuse e gestione dei prompt",           "Trace→Span→Generation, prompt versioning, LLM-as-judge"),
    ("19", "Strategia freelance avanzata",             "Proposta di valore, prezzi, template outreach, ROI"),
    ("20", "Fine-tuning",                               "Albero decisionale, JSONL, distillazione dataset, break-even"),
    ("21", "Sicurezza nei sistemi LLM",                "Prompt injection, PII, rate limiting, guardrail"),
    ("22", "Comunicazione dei progetti",               "Tre documenti di consegna, metriche tradotte, demo"),
    ("23", "Pricing e gestione dei progetti",          "PERT, change request, ambito, salute del progetto"),
    ("24", "Chiusura e lancio",                        "Portfolio, proposta di valore, contatti, roadmap"),
]

PROGRESS_MD = f"""# Piano di formazione AI Engineering — Registro dei progressi

**Stato:** ✅ Completato ({HOY.isoformat()})
**Durata:** 24 settimane
**Obiettivo:** primo cliente freelance come AI Engineer

---

## Riepilogo delle settimane

| # | Tema | Contenuto chiave | Stato |
|---|------|----------------|--------|
""" + "".join(
    f"| {n} | {tema} | {contenido} | ✅ |\n"
    for n, tema, contenido in RESUMEN_SEMANAS
) + """
---

## Progetti costruiti

| Progetto | Stack | Portfolio |
|---------|-------|-----------|
| DocBot RAG API | FastAPI + Chroma + OpenAI + Railway | ✅ pronto |
| ResearchAgent API | FastAPI + DuckDuckGo + BeautifulSoup + Langfuse + guardrail | ✅ pronto |

## Strumenti di business

| Strumento | File |
|------------|--------|
| Calcolatrice day rate | `notebooks/19_estrategia_freelance.ipynb` |
| Generatore di proposte | `notebooks/19_estrategia_freelance.ipynb` |
| Stimatore PERT | `notebooks/23_pricing_gestion_proyectos.ipynb` |
| Generatore di preventivi | `notebooks/23_pricing_gestion_proyectos.ipynb` |
| Template change request | `notebooks/23_pricing_gestion_proyectos.ipynb` |
| Documenti di consegna | `notebooks/22_comunicacion_proyectos.ipynb` |

---

## Prossimo passo

Vedi `lanzamiento/README_portfolio.md` e il piano d'azione in `notebooks/24_cierre_plan_hoja_de_ruta.ipynb`.
"""

(CURSO_DIR / "progress.md").write_text(PROGRESS_MD)
print("✓ progress.md aggiornato con le 24 settimane")
print()
print(f"Totale: {len(RESUMEN_SEMANAS)} settimane completate")

# Inventario finale
dispensas = sorted((CURSO_DIR / "dispensas").glob("*.md"))
notebooks = sorted((CURSO_DIR / "notebooks").glob("*.ipynb"))
print(f"Dispense generate: {len(dispensas)}")
print(f"Notebook generati: {len(notebooks)}")

## Cella 7 — Le tre metriche di successo a sei mesi

Definisci oggi come misurerai se il piano ha funzionato. Senza metriche concrete,
è impossibile sapere se stai avanzando.

In [ ]:
FECHA_REVISION = HOY + timedelta(weeks=26)  # 6 mesi

METRICAS_EXITO = {
    "fecha_revision": FECHA_REVISION.isoformat(),
    "metricas": [
        {
            "metrica":   "Ricavi da progetti IA",
            "objetivo":  "Almeno 1 progetto fatturato",
            "minimo":    "1 progetto pilota firmato (anche piccolo)",
            "como_medir": "Fatture emesse",
        },
        {
            "metrica":   "Portfolio pubblico",
            "objetivo":  "2 progetti pubblicati con README orientato al cliente",
            "minimo":    "1 progetto pubblicato con un caso d'uso reale",
            "como_medir": "URL GitHub accessibile + testimonianza di un cliente",
        },
        {
            "metrica":   "Rete di contatti attiva",
            "objetivo":  "5 conversazioni reali su progetti di IA",
            "minimo":    "2 riunioni di discovery con potenziali clienti",
            "como_medir": "Registro nel CRM (anche solo un foglio di calcolo)",
        },
    ]
}

print(f"METRICHE DI SUCCESSO — revisione il {FECHA_REVISION.strftime('%d %b %Y')}")
print("=" * 60)
for i, m in enumerate(METRICAS_EXITO["metricas"], 1):
    print(f"\n{i}. {m['metrica']}")
    print(f"   Obiettivo: {m['objetivo']}")
    print(f"   Minimo:    {m['minimo']}")
    print(f"   Misura:    {m['como_medir']}")

# Salva le metriche
import json
(LAUNCH_DIR / "metricas_exito.json").write_text(
    json.dumps(METRICAS_EXITO, ensure_ascii=False, indent=2)
)
print(f"\n✓ lanzamiento/metricas_exito.json salvato")
print(f"  Revisione programmata: {FECHA_REVISION.strftime('%d %b %Y')}")

## Cella 8 — Inventario finale e chiusura

In [ ]:
print("╔══════════════════════════════════════════════════════╗")
print("║         PIANO AI ENGINEERING — COMPLETATO             ║")
print("╚══════════════════════════════════════════════════════╝")
print()
print("COSA HAI:")

inventario = [
    ("📚", "24 dispense teoriche",          str(CURSO_DIR / "dispensas")),
    ("💻", "24 notebook di pratica",        str(CURSO_DIR / "notebooks")),
    ("🤖", "DocBot RAG API",                str(PROJECT_DIR / "docbot-api")),
    ("🔍", "ResearchAgent API",             str(PROJECT_DIR / "research-agent")),
    ("💼", "Strumenti di business",         str(CURSO_DIR / "herramientas-negocio")),
    ("🚀", "Materiali di lancio",           str(LAUNCH_DIR)),
]

for emoji, nombre, path in inventario:
    p = Path(path)
    existe = "✓" if p.exists() else "✗"
    print(f"  {existe} {emoji}  {nombre}")

print()
print("COSA MANCA (lo ottieni solo lavorando con clienti reali):")
for item in [
    "Il criterio per dire di no a un progetto impostato male",
    "La velocità: il quinto progetto richiede il tempo che avevi stimato",
    "La fiducia nel prezzo: dire '6.000€' senza esitare",
]:
    print(f"  → {item}")

print()
print(f"Prossima azione: apri lanzamiento/README_portfolio.md")
print(f"             e invia il primo messaggio della lista Tier 1.")
print()
print(f"Data di revisione delle metriche: {FECHA_REVISION.strftime('%d %b %Y')}")

## Sfida finale — La riunione di discovery

Il primo passo con qualsiasi cliente potenziale non è inviare una proposta.
È una riunione di 30 minuti per capire il suo problema prima di proporre qualsiasi cosa.

Progetta una scaletta di cinque domande per quella riunione. Le domande devono:

1. Far sì che il cliente descriva il problema con parole sue
2. Rivelare quanto gli costa il problema (tempo, denaro, rischio)
3. Mostrare cosa ha già provato (per non proporre la stessa cosa)
4. Scoprire chi prende la decisione d'acquisto
5. Aprire la porta alla proposta senza fare pressione

```python
GUION_DISCOVERY = [
    # TODO: scrivi qui le tue cinque domande
    # Non sono domande tecniche — sono domande di business
    # Esempio di ciò che NON va bene: "Usate già un sistema RAG?"
    # Esempio di ciò che va bene: "Quanto tempo dedica una persona del vostro team
    #                              a cercare questa informazione ogni settimana?"
]
```

Una volta pronta la scaletta, esercitati con qualcuno del tuo ambiente
che faccia da cliente. 30 minuti. Cronometrati.